# 🧪 VQA Evaluation Pipeline: Submission vs. Ground Truth

This script performs the heavy lifting of aligning your model's predictions with the actual ground truth and calculating scientific VQA metrics.

In [ ]:
import json
import string
from pathlib import Path
import numpy as np
import codecs

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from rouge_score import rouge_scorer
from bert_score import score as calc_bert_score

In [ ]:
BASE_DIR = Path.cwd().parent
EVAL_CATEGORY = "dev"
EVAL_DATA_DIR = (
    BASE_DIR / "ALD-E-ImageMiner" / "icdar2026-competition-data" / EVAL_CATEGORY
)

SUBMISSION_FILE = BASE_DIR / "submission_dev.json"
RESULTS_FILE = BASE_DIR / "evaluation_results.json"

In [ ]:
def normalize_text(text):
    if not text:
        return ""

    # 1. Handle unicode-escaped sequences
    try:
        text = codecs.decode(text, "unicode_escape")
    except:
        pass

    # 2. Lowercase and strip
    text = text.strip().lower()

    # 3. Basic punctuation removal (as you already have)
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Note: For "unit conversion," you might need specific regex if
    # the task expects "100C" to match "100 degrees celsius"
    return text


def calculate_set_metrics(pred_str, gt_str):
    """Calculates Set-based Precision, Recall, and F1 for List questions."""
    pred_set = set([x.strip().lower() for x in pred_str.split(",") if x.strip()])
    gt_set = set([x.strip().lower() for x in gt_str.split(",") if x.strip()])

    if not gt_set and not pred_set:
        return 1.0, 1.0, 1.0
    if not gt_set or not pred_set:
        return 0.0, 0.0, 0.0

    tp = len(pred_set.intersection(gt_set))
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * (precision * recall) / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return precision, recall, f1

Load Predictions into a searchable dictionary: `preds_dict[sample_id][subfigure][question] = answer`

In [ ]:
preds_dict = {}

with open(SUBMISSION_FILE, "r", encoding="utf-8") as f:
    submission_data = json.load(f)
    for item in submission_data:
        s_id = item.get("sample_id")
        preds_dict[s_id] = {}
        for sub_key, q_list in item.get("vqa", {}).items():
            preds_dict[s_id][sub_key] = {}
            for q_obj in q_list:
                preds_dict[s_id][sub_key][q_obj.get("question", "")] = q_obj.get(
                    "answer", ""
                )

Load Ground Truth and match with Predictions

In [ ]:
predictions = {
    "Yes/No": {"preds": [], "gts": []},
    "Factoid": {"preds": [], "gts": []},
    "List": {"preds": [], "gts": []},
    "Paragraph": {"preds": [], "gts": []},
}

json_files = list(EVAL_DATA_DIR.rglob("*.json"))
for json_file in json_files:
    fullpath = str(json_file)
    if ".vscode" in fullpath or "content" in fullpath:
        continue

    with open(json_file, "r", encoding="utf-8") as f:
        gt_data = json.load(f)

    sample_id = gt_data.get("sample_id")
    if not sample_id or sample_id not in preds_dict:
        continue

    for sub_key, q_list in gt_data.get("vqa", {}).items():
        if sub_key not in preds_dict[sample_id]:
            continue

        for gt_q_obj in q_list:
            q_text = gt_q_obj.get("question") or gt_q_obj.get("questions")
            ans_type = gt_q_obj.get("answer_type", "")
            gt_answer = gt_q_obj.get("answer", "")

            pred_answer = preds_dict[sample_id][sub_key].get(q_text)

            if pred_answer is not None and ans_type in predictions:
                predictions[ans_type]["preds"].append(pred_answer)
                predictions[ans_type]["gts"].append(gt_answer)

In [ ]:
print("\n🧮 Calculating metrics for each answer type...")
results = {}
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# 1. Yes/No
if predictions["Yes/No"]["gts"]:
    preds_yn = [
        1 if "yes" in normalize_text(p) else 0 for p in predictions["Yes/No"]["preds"]
    ]
    gts_yn = [
        1 if "yes" in normalize_text(g) else 0 for g in predictions["Yes/No"]["gts"]
    ]
    acc = accuracy_score(gts_yn, preds_yn)
    p, r, f1, _ = precision_recall_fscore_support(
        gts_yn, preds_yn, average="micro", zero_division=0
    )
    results["Yes/No"] = {"Accuracy": acc, "Precision": p, "Recall": r, "F1-score": f1}
    print("✅ Computed Yes/No metrics.")

# 2. Factoid
if predictions["Factoid"]["gts"]:
    em_scores = [
        1.0 if normalize_text(p) == normalize_text(g) else 0.0
        for p, g in zip(predictions["Factoid"]["preds"], predictions["Factoid"]["gts"])
    ]
    rouge_scores = [
        scorer.score(g, p)["rougeL"].fmeasure
        for p, g in zip(predictions["Factoid"]["preds"], predictions["Factoid"]["gts"])
    ]
    results["Factoid"] = {
        "Exact Match": np.mean(em_scores),
        "ROUGE-L": np.mean(rouge_scores),
    }
    print("📋 Computed Factoid metrics.")

# 3. List (Micro-averaged)
if predictions["List"]["gts"]:
    global_tp, global_fp, global_fn = 0, 0, 0

    for p_str, g_str in zip(predictions["List"]["preds"], predictions["List"]["gts"]):
        # Use your normalization inside the set comprehension
        pred_set = set([normalize_text(x) for x in p_str.split(",") if x.strip()])
        gt_set = set([normalize_text(x) for x in g_str.split(",") if x.strip()])

        # Aggregate counts globally
        global_tp += len(pred_set.intersection(gt_set))
        global_fp += len(pred_set - gt_set)
        global_fn += len(gt_set - pred_set)

    # Calculate global metrics
    p = global_tp / (global_tp + global_fp) if (global_tp + global_fp) > 0 else 0.0
    r = global_tp / (global_tp + global_fn) if (global_tp + global_fn) > 0 else 0.0
    f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

    results["List"] = {
        "Set Precision": p,
        "Set Recall": r,
        "Set F1": f1,
        "Total_TP": global_tp,  # Useful for debugging
    }
    print("📜 Computed Micro-averaged List metrics.")

# 4. Paragraph
if predictions["Paragraph"]["gts"]:
    preds_para = predictions["Paragraph"]["preds"]
    gts_para = predictions["Paragraph"]["gts"]
    rouge_scores = [
        scorer.score(g, p)["rougeL"].fmeasure for p, g in zip(preds_para, gts_para)
    ]
    P, R, F1 = calc_bert_score(
        preds_para,
        gts_para,
        lang="en",
        verbose=False,
        model_type="distilbert-base-uncased",
    )
    results["Paragraph"] = {
        "ROUGE-L": np.mean(rouge_scores),
        "BERTScore F1": F1.mean().item(),
    }
    print("📝 Computed Paragraph metrics.")

Append the evaluation metrics to the JSON file to keep a running history

In [ ]:
if RESULTS_FILE.exists():
    with open(RESULTS_FILE, "r") as f:
        try:
            history = json.load(f)
        except json.JSONDecodeError:
            history = []
else:
    history = []

history.append(
    {
        "run_type": "submission_evaluation",
        "submission_file": SUBMISSION_FILE.name,
        "metrics": results,
    }
)

with open(RESULTS_FILE, "w") as f:
    json.dump(history, f, indent=4)

print(f"\n✨ Evaluation complete! State updated in: {RESULTS_FILE}")
print(json.dumps(results, indent=2))